<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/custom_Yolo11Checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
    %pip install sahi ultralytics

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image
from ultralytics import YOLO
from PIL import Image as PILImage, ImageDraw, ImageFont
import numpy as np
import cv2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.12.0.88
    Uninstalling opencv-python-4.12.0.88:
      Successfully uninstalled opencv-python-4.12.0.88
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLO11n model
model = YOLO("yolo11n.pt")

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def mask_water_area(img):
    """
    Mask out the left-bottom corner (land, parking lot, trees)
    and keep the water area.
    """
    h, w = img.shape[:2]

    # Create empty mask
    mask = np.zeros((h, w), dtype=np.uint8)

    # Define polygon for the area to remove
    mask_polygon = np.array([
        [0, int(h * 0.40)],        # left mid
        [int(w * 0.20), int(h * 0.40)],
        [int(w * 0.80), h],        # bottom
        [0, h]
    ], dtype=np.int32)

    # Fill polygon (white = area to remove)
    cv2.fillPoly(mask, [mask_polygon], 255)

    # Invert mask to keep water area
    mask = cv2.bitwise_not(mask)

    # Apply mask
    masked_img = cv2.bitwise_and(img, img, mask=mask)

    return masked_img

# -------------------------------
# Example: Process all images in a folder
# -------------------------------
input_folder = '/content/Jupiter_Inlet/'
output_folder = '/content/Masked_Images/'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Failed to load {filename}")
            continue

        masked_img = mask_water_area(img)

        # Save masked image
        cv2.imwrite(os.path.join(output_folder, filename), masked_img)

        # Optional: show the first image only
        # plt.figure(figsize=(10, 6))
        # plt.imshow(cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB))
        # plt.title(f"Masked: {filename}")
        # plt.axis("off")
        # plt.show()

In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import csv

# -------------------------------
# SETTINGS
# -------------------------------
input_folder = '/content/Masked_Images/'      # folder with masked images
output_folder = '/content/Annotated/'        # folder to save annotated images
os.makedirs(output_folder, exist_ok=True)

csv_file = '/content/boat_counts.csv'        # CSV to save counts

# Load your custom YOLOv11 model
model = YOLO('weights.pt')  # <-- your Roboflow weights file

# -------------------------------
# PROCESS IMAGES
# -------------------------------
results_list = []

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Apply mask
        masked_img = mask_water_area(img)

        # Run YOLOv11 inference (increase resolution if boats are small)
        results = model.predict(masked_img, imgsz=1280)  # imgsz larger helps small boats

        # Count boats
        boat_boxes = [
            box for box, cls in zip(results[0].boxes.xyxy, results[0].boxes.cls)
            if results[0].names[int(cls)].lower() == 'boat'
        ]
        num_boats = len(boat_boxes)
        print(f"{filename}: {num_boats} boat(s) detected")

        # Save annotated image
        annotated_img = results[0].plot()
        cv2.imwrite(os.path.join(output_folder, filename), annotated_img)

        # Append results to list
        results_list.append([filename, num_boats])

# -------------------------------
# SAVE CSV
# -------------------------------
with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Filename', 'NumBoats'])
    writer.writerows(results_list)

print(f"Processing complete. Annotated images saved in {output_folder} and counts saved to {csv_file}")


0: 960x1280 1 Boat, 2337.1ms
Speed: 32.0ms preprocess, 2337.1ms inference, 66.3ms postprocess per image at shape (1, 3, 960, 1280)
o080716r.jpg: 1 boat(s) detected

0: 960x1280 (no detections), 2231.8ms
Speed: 18.5ms preprocess, 2231.8ms inference, 1.1ms postprocess per image at shape (1, 3, 960, 1280)
o071727k.jpg: 0 boat(s) detected

0: 960x1280 (no detections), 2171.6ms
Speed: 23.4ms preprocess, 2171.6ms inference, 1.3ms postprocess per image at shape (1, 3, 960, 1280)
s300703s.jpg: 0 boat(s) detected

0: 960x1280 (no detections), 1701.8ms
Speed: 15.7ms preprocess, 1701.8ms inference, 1.1ms postprocess per image at shape (1, 3, 960, 1280)
o020600v.jpg: 0 boat(s) detected

0: 960x1280 1 Boat, 1644.0ms
Speed: 14.9ms preprocess, 1644.0ms inference, 1.2ms postprocess per image at shape (1, 3, 960, 1280)
o120840j.jpg: 1 boat(s) detected

0: 960x1280 3 Boats, 1784.5ms
Speed: 15.2ms preprocess, 1784.5ms inference, 1.3ms postprocess per image at shape (1, 3, 960, 1280)
o101422s.jpg: 3 boat